In [2]:
# library imports
import os
import glob
import json
import pandas as pd

In [ ]:
base_path = "../../../prompting results/02_final_results"

# Find all files with extracted_output (aka files beginning with analyzed_) for checking
all_files = glob.glob(os.path.join(base_path, "**", "analyzed_*.json"), recursive=True)

print(f"Found {len(all_files)} files.") # print total file count [should be 480: 3 per model x 5 models (gpt high and minimal accounting for 2) x 32 questions)

# read ALL loop
rows = []

for file in all_files:
    with open(file, "r") as f:
        data = json.load(f)
        # each JSON file is a list of entries
        rows.extend(data)

print(f"Loaded {len(rows)} total entries.") # check on total rows (should be 811,560 -> see prompt_generatio_val for more details)
# put it all together
whole_df = pd.DataFrame(rows)

# print column names
print("\nColumns in DataFrame:")
print(whole_df.columns.tolist())

Found 480 files.
Loaded 811560 total entries.

Columns in DataFrame:
['iteration', 'model_type', 'reason_effort', 'poll_name', 'original_question', 'demographics', 'embody', 'prompt', 'output', 'response_options', 'extracted_option']


In [11]:
# Check for duplicate file paths
print("Total files found:", len(all_files))
print("Unique file paths:", len(set(all_files)))

Total files found: 480
Unique file paths: 480


In [14]:
# Rows per model
rows_per_model = (
    whole_df
    .groupby("model_type")
    .size()
    .reset_index(name="n rows")
    .sort_values("n rows", ascending=False)
)

print("Rows per model:")
print(rows_per_model) 
# we expect 162,312 for deepseek, llama, and mistral (54,104 total prompts x 3 iterations) 
# we expect double that number for gpt-nano; both assumptions are satisfied

Rows per model:
                                  model_type  n rows
1                                 gpt-5-nano  324624
0                  deepseek-ai/DeepSeek-V3.2  162312
2      meta-llama/Meta-Llama-3.1-8B-Instruct  162312
3  mistralai/Mistral-Small-24B-Instruct-2501  162312


In [15]:
rows_per_question = (
    whole_df
    .groupby(["poll_name"])
    .size()
    .reset_index(name="n_rows")
    .sort_values(["n_rows", "poll_name"])
)

print("\nRows per model x poll x iteration:")
print(rows_per_question)


Rows per model x poll x iteration:
                                  poll_name  n_rows
13              DUKE_PRESS_988_Awareness_Q1    1095
4   Biden_and_Trump_Handling_of_Problems_Q1    6495
5   Biden_and_Trump_Handling_of_Problems_Q2    6495
6   Biden_and_Trump_Handling_of_Problems_Q3    6495
0                     AFSP_Mental_Health_Q1   12975
1                     AFSP_Mental_Health_Q2   12975
2                     AFSP_Mental_Health_Q3   12975
3                     AFSP_Mental_Health_Q4   12975
7                      DATA_FOR_PROGRESS_Q1   12975
8                      DATA_FOR_PROGRESS_Q2   12975
9                      DATA_FOR_PROGRESS_Q3   12975
10                     DATA_FOR_PROGRESS_Q4   12975
11                     DATA_FOR_PROGRESS_Q5   12975
12                     DATA_FOR_PROGRESS_Q6   12975
29                    Suicide_prevention_Q1   25935
30                    Suicide_prevention_Q2   25935
31                    Suicide_prevention_Q3   25935
14                IPSOS_KP_9

Total prompts per question: cross product * 2(embody and specialist) + 1 (direct prompt) * 3 (iterations) * 5 (models)

cross products (math to validate these found at "notebooks/00_preliminary_testing_validation/code_validation/prompt_generation_val.ipynb"):
- AFSP (no party): 3 * 3 * 4 * 3 * 4  = 432 each
- Duke (no income or  age): 3 * 3 * 4  = 36 each
- Science Direct: 3 * 3 * 4 * 3 * 4 * 3 = 1,296 each 
- Ipsos: 3 * 3 * 4 * 3 * 4 * 3 =1,296 each
- Progress (no income): 3 * 3 * 4 * 4 * 3  = 432 each
- YouGov BT (no edu): 3 * 2 (non bin rmv) * 4 * 3 * 3 = 216 each
- YouGov SP: 3* 2 (non bin rmv) * 4 * 3 * 3 = 432 each

Total expected prompts:
Duke: 36 * 2 + 1 * 3 * 5 = 1,095
AFSP: 432 * 2 + 1 * 3 * 5 = 12975 per Q
Science Direct: 1,296 * 2 + 1 * 3 * 5 = 38,895 per Q
Ipsos: 1,296 * 2 + 1 * 3 * 5 = 38,895 per Q
Progress: 423 * 2 + 1 * 3 * 5 = 12,975 per Q
You Gov: 216 * 2 + 1 * 3 * 5 = 6,495 per Q

Results match math.
